# 07 — LSTM hurdle v2

Это production-версия LSTM без смешивания с другими моделями.

Главные изменения:

1. **Padding исправлен:** модель знает реальную длину истории; sequence compacted + `pack_padded_sequence`, pooling только по валидным дням.
2. Убран `BatchNorm` по sequence/time; вместо него normalization внутри projection/head, не зависящий от состава batch.
3. Добавлен **`user_id` embedding** как panel/fixed-effect сигнал.
4. Добавлена отдельная **short-history summary branch**: 1/3/7/14/30/60/90-day intent, recency и recent-vs-previous deltas.
5. Число эпох classifier/regressor выбирается по последнему temporal holdout **по итоговому RMSLE hurdle**, а затем модели переобучаются на всех labeled cutoff ровно столько эпох.
6. Все веса, preprocessing stats, config, data-meta и user vocabulary сохраняются в `models/lstm_hurdle/`.
7. Submission получает уникальное имя; в конце есть reload smoke-test сохранённых весов.

Hurdle остаётся раздельным — это специально более консервативный апгрейд сильной текущей модели:

$$
\widehat{\log(1+y)} = \sigma(z_{cls}) \cdot \max(0, z_{reg}).
$$

In [1]:
from pathlib import Path
from copy import deepcopy
import hashlib
import json
import random
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def find_project_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data" / "lstm" / "meta.json").exists():
            return path
    raise FileNotFoundError("Сначала запустите 06_LSTM_Data_Preparation.ipynb")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

if META.get("format_version") != 2:
    raise RuntimeError(
        "Нужен LSTM data format v2. Запустите новый 06_LSTM_Data_Preparation.ipynb."
    )

LABELED_CUTOFFS = META["labeled_cutoffs"]
INFERENCE_CUTOFF = META["inference_cutoff"]
BASE_SEQUENCE_FEATURES = META["base_sequence_features"]
CALENDAR_FEATURES = META["calendar_sequence_features"]
STATIC_FEATURES = META["static_features"]
STATIC_LOG_FEATURES = META["static_log_copy_features"]
SEQ_LEN = int(META["seq_len"])
N_USERS = int(META["user_count"])

BASE_INDEX = {name: i for i, name in enumerate(BASE_SEQUENCE_FEATURES)}
STATIC_LOG_INDICES = [STATIC_FEATURES.index(name) for name in STATIC_LOG_FEATURES]

HOLDOUT_TRAIN_CUTOFFS = LABELED_CUTOFFS[:-1]
HOLDOUT_CUTOFF = LABELED_CUTOFFS[-1]

BATCH_SIZE = 1024
MAX_CLASSIFIER_EPOCHS = 6
MAX_REGRESSOR_EPOCHS = 16
LR = 5e-4
WEIGHT_DECAY = 5e-4
RANDOM_STATE = 42
USER_EMBED_DIM = 16

# Если потом захотите seed ensemble той же LSTM, можно поставить [42, 123, 2026].
# Сейчас оставляю один seed, чтобы один run не умножал время обучения на 3.
FINAL_SEEDS = [42]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
NUM_WORKERS = 4 if DEVICE.type == "cuda" else 0


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print("device:", DEVICE)
print("holdout:", HOLDOUT_CUTOFF)
print("users in embedding:", N_USERS)

device: mps
holdout: 2026-01-14
users in embedding: 250000


## Dataset

`history_length` и `user_index` приходят из `06`.

`history_length` — число календарных дней, которые реально существуют для пользователя внутри 90-дневного окна. Оно отличается от `active`: пользователь может существовать, но быть неактивным в конкретный день.

In [2]:
class HybridDataset(Dataset):
    def __init__(self, cutoff, with_target=True, positive_only=False):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.static = np.load(path / "static.npy", mmap_mode="r")
        self.calendar = np.load(path / "calendar.npy", mmap_mode="r").astype(np.float32)
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.user_index = np.load(path / "user_index.npy", mmap_mode="r")
        self.history_length = np.load(path / "history_length.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None
        self.indices = np.flatnonzero(self.y > 0) if positive_only else None

    def __len__(self):
        return len(self.X) if self.indices is None else len(self.indices)

    def __getitem__(self, i):
        j = i if self.indices is None else int(self.indices[i])

        sequence = np.concatenate([
            np.asarray(self.X[j], dtype=np.float32),
            self.calendar,
        ], axis=1)
        static = np.array(self.static[j], dtype=np.float32)
        user_index = int(self.user_index[j])
        history_length = int(self.history_length[j])

        sequence = torch.from_numpy(sequence)
        static = torch.from_numpy(static)
        user_index = torch.tensor(user_index, dtype=torch.long)
        history_length = torch.tensor(history_length, dtype=torch.long)

        if self.y is None:
            return sequence, static, user_index, history_length

        y = torch.tensor(float(self.y[j]), dtype=torch.float32)
        return sequence, static, user_index, history_length, y


def make_loader(cutoffs, shuffle=False, with_target=True, positive_only=False):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]

    dataset = ConcatDataset([
        HybridDataset(
            cutoff,
            with_target=with_target,
            positive_only=positive_only,
        )
        for cutoff in cutoffs
    ])

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )

## Sequence features

На батче добавляются:

- history-availability mask;
- `has_*` и funnel ratios;
- masked rolling mean на 3/7/14/30 дней;
- masked activity rate;
- первые разности без искусственного скачка на первом доступном дне.

Отдельная summary-ветка получает 1/3/7/14/30/60/90-day суммы, recency и recent-vs-previous deltas. Это даёт сети явный short-term intent, не заставляя LSTM заново вычислять все агрегаты из скрытого состояния.

In [3]:
def history_mask_from_lengths(lengths, steps=SEQ_LEN):
    positions = torch.arange(steps, device=lengths.device).unsqueeze(0)
    starts = steps - lengths.unsqueeze(1)
    return positions >= starts


def causal_masked_mean(values, valid_mask, window):
    values = values * valid_mask
    num = F.avg_pool1d(
        F.pad(values.unsqueeze(1), (window - 1, 0)),
        window,
        stride=1,
    ).squeeze(1) * window
    den = F.avg_pool1d(
        F.pad(valid_mask.unsqueeze(1), (window - 1, 0)),
        window,
        stride=1,
    ).squeeze(1) * window
    return num / den.clamp_min(1.0)


def first_difference(values, valid_mask):
    previous_valid = F.pad(valid_mask[:, :-1], (1, 0))
    both_valid = valid_mask * previous_valid
    diff = F.pad(values[:, 1:] - values[:, :-1], (1, 0))
    return diff * both_valid


def safe_ratio(numerator, denominator, max_value=5.0):
    ratio = numerator / denominator.clamp_min(1e-3)
    ratio = torch.where(denominator > 0, ratio, torch.zeros_like(ratio))
    return ratio.clamp(0, max_value)


def make_sequence_features(x, lengths):
    valid_mask = history_mask_from_lengths(lengths, x.shape[1]).float()

    searches_log = x[..., BASE_INDEX["searches"]]
    search_to_cart_log = x[..., BASE_INDEX["search_to_cart"]]
    search_to_ord_log = x[..., BASE_INDEX["search_to_ord"]]
    cat_to_cart_log = x[..., BASE_INDEX["cat_to_cart"]]
    cat_to_ord_log = x[..., BASE_INDEX["cat_to_ord"]]
    to_cart_log = x[..., BASE_INDEX["to_cart"]]
    to_ord_log = x[..., BASE_INDEX["to_ord"]]
    gmv_search_log = x[..., BASE_INDEX["gmv_search"]]
    gmv_log = x[..., BASE_INDEX["gmv"]]
    active = x[..., BASE_INDEX["active"]]

    searches = torch.expm1(searches_log).clamp_min(0)
    search_to_cart = torch.expm1(search_to_cart_log).clamp_min(0)
    search_to_ord = torch.expm1(search_to_ord_log).clamp_min(0)
    cat_to_cart = torch.expm1(cat_to_cart_log).clamp_min(0)
    cat_to_ord = torch.expm1(cat_to_ord_log).clamp_min(0)
    to_cart = torch.expm1(to_cart_log).clamp_min(0)
    to_ord = torch.expm1(to_ord_log).clamp_min(0)
    gmv_search = torch.expm1(gmv_search_log).clamp_min(0)
    gmv = torch.expm1(gmv_log).clamp_min(0)

    derived = [
        valid_mask,
        (search_to_cart > 0).float(),
        (search_to_ord > 0).float(),
        (cat_to_cart > 0).float(),
        (cat_to_ord > 0).float(),
        safe_ratio(search_to_cart, searches),
        safe_ratio(search_to_ord, searches),
        safe_ratio(to_ord, to_cart),
        safe_ratio(gmv_search, gmv, 1.5),
    ]

    for values in [searches_log, to_cart_log, to_ord_log, gmv_log]:
        for window in [3, 7, 14, 30]:
            derived.append(causal_masked_mean(values, valid_mask, window))

    for window in [3, 7, 14, 30]:
        derived.append(causal_masked_mean(active, valid_mask, window))

    for values in [searches_log, to_ord_log, gmv_log]:
        derived.append(first_difference(values, valid_mask))

    return torch.cat([x, torch.stack(derived, dim=-1)], dim=-1)


def _window_raw_sum(x, feature_name, window):
    values = torch.expm1(x[..., BASE_INDEX[feature_name]]).clamp_min(0)
    return values[:, -window:].sum(dim=1)


def _days_since_event(x, lengths, feature_name):
    values = torch.expm1(x[..., BASE_INDEX[feature_name]]).clamp_min(0)
    valid = history_mask_from_lengths(lengths, x.shape[1])
    event = (values > 0) & valid
    positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0).expand_as(values)
    last = torch.where(event, positions, torch.full_like(positions, -1)).amax(dim=1)
    days = (x.shape[1] - 1 - last).float()
    # no event in available history -> отдельное большое, но ограниченное значение
    return torch.where(last >= 0, days / x.shape[1], torch.full_like(days, 1.25))


def make_short_summary(x, lengths):
    windows = [1, 3, 7, 14, 30, 60, 90]
    volume_names = ["searches", "to_cart", "to_ord", "gmv"]
    features = []

    for name in volume_names:
        for window in windows:
            features.append(torch.log1p(_window_raw_sum(x, name, window)))

    active = x[..., BASE_INDEX["active"]]
    for window in windows:
        exposure = lengths.clamp(max=window).float()
        features.append(active[:, -window:].sum(dim=1) / exposure.clamp_min(1.0))

    for name in ["searches", "to_cart", "to_ord", "gmv"]:
        features.append(_days_since_event(x, lengths, name))

    for name in volume_names:
        for window in [3, 7, 30]:
            recent = _window_raw_sum(x, name, window)
            if 2 * window <= x.shape[1]:
                values = torch.expm1(x[..., BASE_INDEX[name]]).clamp_min(0)
                previous = values[:, -2 * window:-window].sum(dim=1)
            else:
                previous = torch.zeros_like(recent)
            features.append(torch.log1p(recent) - torch.log1p(previous))

    features.append(lengths.float() / x.shape[1])
    return torch.stack(features, dim=1)


# 1 mask + 4 has + 4 ratios + 4*4 rolling + 4 activity + 3 diffs
N_DERIVED_SEQUENCE = 1 + 4 + 4 + 16 + 4 + 3
SEQ_INPUT_SIZE = len(BASE_SEQUENCE_FEATURES) + len(CALENDAR_FEATURES) + N_DERIVED_SEQUENCE
SHORT_SUMMARY_SIZE = 4 * 7 + 7 + 4 + 4 * 3 + 1

print("sequence features:", SEQ_INPUT_SIZE)
print("short summary:", SHORT_SUMMARY_SIZE)

sequence features: 51
short summary: 52


## Static preprocessing

Статистика fit'ится только по train cutoff. Missing-mask остаётся отдельным признаком.

In [4]:
STATIC_INPUT_SIZE = 2 * len(STATIC_FEATURES) + len(STATIC_LOG_INDICES)


def augment_static_numpy(raw):
    raw = np.asarray(raw, dtype=np.float32)
    source = raw[:, STATIC_LOG_INDICES]

    logs = np.where(
        np.isfinite(source),
        np.log1p(np.clip(source, 0, None)),
        np.nan,
    ).astype(np.float32)

    missing = (~np.isfinite(raw)).astype(np.float32)
    return np.concatenate([raw, logs, missing], axis=1)


def fit_static_stats(cutoffs, chunk_size=65_536):
    sums = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    sums_sq = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    counts = np.zeros(STATIC_INPUT_SIZE, dtype=np.int64)

    for cutoff in cutoffs:
        raw = np.load(DATA_DIR / cutoff / "static.npy", mmap_mode="r")

        for start in range(0, len(raw), chunk_size):
            block = augment_static_numpy(raw[start:start + chunk_size])
            finite = np.isfinite(block)
            safe = np.where(finite, block, 0.0).astype(np.float64)

            sums += safe.sum(axis=0)
            sums_sq += (safe * safe).sum(axis=0)
            counts += finite.sum(axis=0)

    counts = np.maximum(counts, 1)
    mean = sums / counts
    std = np.sqrt(np.maximum(sums_sq / counts - mean * mean, 1e-6))

    return (
        torch.tensor(mean, dtype=torch.float32),
        torch.tensor(std, dtype=torch.float32),
    )


def normalize_static(raw, stats):
    source = raw[:, STATIC_LOG_INDICES]
    logs = torch.where(
        torch.isfinite(source),
        torch.log1p(source.clamp_min(0)),
        torch.nan,
    )

    augmented = torch.cat([
        raw,
        logs,
        (~torch.isfinite(raw)).float(),
    ], dim=1)

    mean, std = stats
    mean = mean.to(raw.device)
    std = std.to(raw.device)

    augmented = torch.where(torch.isfinite(augmented), augmented, mean)
    return (augmented - mean) / std


print("static features after augmentation:", STATIC_INPUT_SIZE)

static features after augmentation: 257


## Архитектура v2

Изменения относительно старой:

```text
sequence
  -> derived causal features
  -> remove left padding / pack
  -> Linear + LayerNorm
  -> 2-layer BiLSTM
  -> final hidden + masked mean + masked max
  -> 256

short-history summary -> 64
static normalized      -> 128
user embedding         -> 16

concat -> fusion MLP -> scalar
```

`BatchNorm` убран: при variable-length zero-heavy sequences он зависел от количества padding в конкретном batch.

In [5]:
def compact_left_padded(sequence, lengths):
    """Переносит валидный хвост [padding | history] в начало [history | padding]."""
    batch, steps, channels = sequence.shape
    positions = torch.arange(steps, device=sequence.device).unsqueeze(0).expand(batch, -1)
    starts = steps - lengths.unsqueeze(1)
    source_pos = (positions + starts).clamp(max=steps - 1)
    gathered = sequence.gather(
        1,
        source_pos.unsqueeze(-1).expand(-1, -1, channels),
    )
    valid = positions < lengths.unsqueeze(1)
    return gathered * valid.unsqueeze(-1)


class HybridLSTM(nn.Module):
    def __init__(
        self,
        n_users,
        user_embed_dim=USER_EMBED_DIM,
        hidden_size=128,
        num_layers=2,
        lstm_dropout=0.25,
        head_dropout=0.30,
    ):
        super().__init__()
        self.hidden_size = hidden_size

        self.sequence_projection = nn.Sequential(
            nn.Linear(SEQ_INPUT_SIZE, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.lstm = nn.LSTM(
            input_size=96,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.sequence_head = nn.Sequential(
            nn.LayerNorm(hidden_size * 6),
            nn.Linear(hidden_size * 6, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
        )

        self.summary_head = nn.Sequential(
            nn.LayerNorm(SHORT_SUMMARY_SIZE),
            nn.Linear(SHORT_SUMMARY_SIZE, 96),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(96, 64),
            nn.GELU(),
        )

        self.static_head = nn.Sequential(
            nn.Linear(STATIC_INPUT_SIZE, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.20),
        )

        self.user_embedding = nn.Embedding(n_users, user_embed_dim)
        nn.init.normal_(self.user_embedding.weight, mean=0.0, std=0.02)
        self.user_dropout = nn.Dropout(0.15)

        fusion_size = 256 + 64 + 128 + user_embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )

    def forward(self, sequence, static, user_index, history_length):
        short_summary = make_short_summary(sequence, history_length)

        sequence = make_sequence_features(sequence, history_length)
        sequence = compact_left_padded(sequence, history_length)
        sequence = self.sequence_projection(sequence)

        # Packed LSTM исключает right padding из recurrent state.
        packed = pack_padded_sequence(
            sequence,
            history_length.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, (hidden, _) = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=SEQ_LEN,
        )

        positions = torch.arange(SEQ_LEN, device=output.device).unsqueeze(0)
        valid = positions < history_length.unsqueeze(1)
        valid_f = valid.unsqueeze(-1)

        pooled_mean = (output * valid_f).sum(dim=1) / history_length.float().unsqueeze(1)
        pooled_max = output.masked_fill(~valid_f, float("-inf")).amax(dim=1)
        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)

        sequence_repr = self.sequence_head(torch.cat([
            last_hidden,
            pooled_mean,
            pooled_max,
        ], dim=1))
        summary_repr = self.summary_head(short_summary)
        static_repr = self.static_head(static)
        user_repr = self.user_dropout(self.user_embedding(user_index))

        fused = torch.cat([
            sequence_repr,
            summary_repr,
            static_repr,
            user_repr,
        ], dim=1)
        return self.fusion(fused).squeeze(1)


MODEL_KWARGS = {
    "n_users": N_USERS,
    "user_embed_dim": USER_EMBED_DIM,
    "hidden_size": 128,
    "num_layers": 2,
    "lstm_dropout": 0.25,
    "head_dropout": 0.30,
}

probe = HybridLSTM(**MODEL_KWARGS)
print("parameters:", f"{sum(p.numel() for p in probe.parameters()):,}")
del probe

parameters: 5,077,193


## Train / predict helpers

Classifier: BCE по `y > 0`.

Regressor: MSE для `log1p(y)` только на `y > 0`.

На holdout сохраняем predictions **каждой эпохи**, после чего выбираем пару `(classifier_epoch, regressor_epoch)` по итоговому hurdle RMSLE. Это лучше, чем независимо выбирать BCE/MSE и потом случайно получить плохую склейку.

In [6]:
def unpack_batch(batch, with_target=True):
    if with_target:
        sequence, static, user_index, history_length, y = batch
        y = y.to(DEVICE, non_blocking=True)
    else:
        sequence, static, user_index, history_length = batch
        y = None

    return (
        sequence.to(DEVICE, non_blocking=True),
        static.to(DEVICE, non_blocking=True),
        user_index.to(DEVICE, non_blocking=True),
        history_length.to(DEVICE, non_blocking=True),
        y,
    )


def train_one_epoch(model, loader, optimizer, task, static_stats):
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        sequence, static, user_index, history_length, y = unpack_batch(batch)
        static = normalize_static(static, static_stats)

        optimizer.zero_grad(set_to_none=True)
        output = model(sequence, static, user_index, history_length)

        if task == "classifier":
            loss = F.binary_cross_entropy_with_logits(output, (y > 0).float())
        else:
            loss = F.mse_loss(output, torch.log1p(y))

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        total_n += len(y)

    return total_loss / max(total_n, 1)


@torch.no_grad()
def predict_raw(model, cutoff, static_stats, with_target=True):
    dataset = HybridDataset(cutoff, with_target=with_target)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )

    model.eval()
    outputs = []
    targets = []

    for batch in loader:
        sequence, static, user_index, history_length, y = unpack_batch(
            batch,
            with_target=with_target,
        )
        static = normalize_static(static, static_stats)
        outputs.append(
            model(sequence, static, user_index, history_length).cpu().numpy()
        )
        if with_target:
            targets.append(y.cpu().numpy())

    target = np.concatenate(targets) if with_target else None
    return np.asarray(dataset.users), np.concatenate(outputs), target


def make_prediction(classifier_logits, regressor_log):
    probability = 1.0 / (1.0 + np.exp(-np.clip(classifier_logits, -30, 30)))
    pred_log = probability * np.clip(regressor_log, 0, None)
    return pred_log, np.expm1(pred_log)


def rmsle_from_log(y, pred_log):
    return float(np.sqrt(np.mean((pred_log - np.log1p(y)) ** 2)))


def fit_with_epoch_predictions(task, train_cutoffs, valid_cutoff, static_stats, max_epochs, seed):
    set_seed(seed)
    loader = make_loader(
        train_cutoffs,
        shuffle=True,
        positive_only=task == "regressor",
    )

    model = HybridLSTM(**MODEL_KWARGS).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max_epochs,
        eta_min=1e-6,
    )

    epoch_outputs = []
    y_valid = None

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, loader, optimizer, task, static_stats)
        scheduler.step()

        _, valid_output, y_valid = predict_raw(
            model,
            valid_cutoff,
            static_stats,
            with_target=True,
        )
        epoch_outputs.append(valid_output.astype(np.float32))
        print(
            f"{task:10s} | epoch {epoch:02d}/{max_epochs:02d} "
            f"| train_loss={train_loss:.5f}"
        )

    return model, epoch_outputs, y_valid


def fit_fixed_epochs(task, cutoffs, static_stats, epochs, seed):
    set_seed(seed)
    loader = make_loader(
        cutoffs,
        shuffle=True,
        positive_only=task == "regressor",
    )

    model = HybridLSTM(**MODEL_KWARGS).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=1e-6,
    )

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, loader, optimizer, task, static_stats)
        scheduler.step()
        print(f"{task:10s} | epoch {epoch:02d}/{epochs:02d} | loss={loss:.5f}")

    return model

## Temporal holdout: выбор числа эпох

Последний labeled cutoff (`2026-01-14`) не участвует в fit весов на этом этапе. Мы используем его только для выбора **двух целых чисел — числа эпох** под конечный hurdle RMSLE.

In [7]:
holdout_static_stats = fit_static_stats(HOLDOUT_TRAIN_CUTOFFS)

holdout_classifier, classifier_epoch_logits, y_holdout = fit_with_epoch_predictions(
    "classifier",
    HOLDOUT_TRAIN_CUTOFFS,
    HOLDOUT_CUTOFF,
    holdout_static_stats,
    MAX_CLASSIFIER_EPOCHS,
    RANDOM_STATE,
)

holdout_regressor, regressor_epoch_logs, y_holdout_reg = fit_with_epoch_predictions(
    "regressor",
    HOLDOUT_TRAIN_CUTOFFS,
    HOLDOUT_CUTOFF,
    holdout_static_stats,
    MAX_REGRESSOR_EPOCHS,
    RANDOM_STATE,
)

np.testing.assert_allclose(y_holdout, y_holdout_reg)

rows = []
for cls_epoch, logits in enumerate(classifier_epoch_logits, start=1):
    for reg_epoch, reg_log in enumerate(regressor_epoch_logs, start=1):
        pred_log, _ = make_prediction(logits, reg_log)
        rows.append({
            "classifier_epoch": cls_epoch,
            "regressor_epoch": reg_epoch,
            "rmsle": rmsle_from_log(y_holdout, pred_log),
        })

epoch_search = pd.DataFrame(rows).sort_values("rmsle", ignore_index=True)
display(epoch_search.head(15))

BEST_CLASSIFIER_EPOCH = int(epoch_search.loc[0, "classifier_epoch"])
BEST_REGRESSOR_EPOCH = int(epoch_search.loc[0, "regressor_epoch"])
HOLDOUT_RMSLE = float(epoch_search.loc[0, "rmsle"])

print("best classifier epoch:", BEST_CLASSIFIER_EPOCH)
print("best regressor epoch:", BEST_REGRESSOR_EPOCH)
print("holdout RMSLE:", f"{HOLDOUT_RMSLE:.6f}")

if BEST_CLASSIFIER_EPOCH == MAX_CLASSIFIER_EPOCHS:
    print("WARNING: classifier optimum on boundary; consider increasing MAX_CLASSIFIER_EPOCHS")
if BEST_REGRESSOR_EPOCH == MAX_REGRESSOR_EPOCHS:
    print("WARNING: regressor optimum on boundary; consider increasing MAX_REGRESSOR_EPOCHS")

classifier | epoch 01/06 | train_loss=0.47649
classifier | epoch 02/06 | train_loss=0.41542
classifier | epoch 03/06 | train_loss=0.36652
classifier | epoch 04/06 | train_loss=0.33154
classifier | epoch 05/06 | train_loss=0.30425
classifier | epoch 06/06 | train_loss=0.28766
regressor  | epoch 01/16 | train_loss=1.49146
regressor  | epoch 02/16 | train_loss=1.25726
regressor  | epoch 03/16 | train_loss=1.13297
regressor  | epoch 04/16 | train_loss=1.06260
regressor  | epoch 05/16 | train_loss=1.01916
regressor  | epoch 06/16 | train_loss=0.98571
regressor  | epoch 07/16 | train_loss=0.95863
regressor  | epoch 08/16 | train_loss=0.93546
regressor  | epoch 09/16 | train_loss=0.91455
regressor  | epoch 10/16 | train_loss=0.89711
regressor  | epoch 11/16 | train_loss=0.88043
regressor  | epoch 12/16 | train_loss=0.86661
regressor  | epoch 13/16 | train_loss=0.85366
regressor  | epoch 14/16 | train_loss=0.84439
regressor  | epoch 15/16 | train_loss=0.83739
regressor  | epoch 16/16 | train_l

,classifier_epoch,regressor_epoch,rmsle
0,1,2,1.704478
1,2,2,1.709014
2,1,1,1.714756
3,1,6,1.718035
4,1,5,1.718172
5,2,5,1.719068
6,2,6,1.719195
7,2,1,1.719470
8,1,4,1.719491
9,2,4,1.719918


best classifier epoch: 1
best regressor epoch: 2
holdout RMSLE: 1.704478


## Финальное обучение на всех labeled cutoff

После выбора epoch-count Jan holdout возвращается в train. Inference cutoff нигде не участвует в fitting.

In [8]:
del holdout_classifier, holdout_regressor
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

final_static_stats = fit_static_stats(LABELED_CUTOFFS)

final_models = []
for seed in FINAL_SEEDS:
    print("\n=== FINAL SEED", seed, "===")
    classifier = fit_fixed_epochs(
        "classifier",
        LABELED_CUTOFFS,
        final_static_stats,
        BEST_CLASSIFIER_EPOCH,
        seed,
    )
    regressor = fit_fixed_epochs(
        "regressor",
        LABELED_CUTOFFS,
        final_static_stats,
        BEST_REGRESSOR_EPOCH,
        seed,
    )
    final_models.append((seed, classifier, regressor))


=== FINAL SEED 42 ===
classifier | epoch 01/01 | loss=0.47534
regressor  | epoch 01/02 | loss=1.46568
regressor  | epoch 02/02 | loss=1.22576


## Сохранение весов и полного inference-контракта

После этой cell папка `models/lstm_hurdle/` самодостаточна относительно архитектуры/весов/preprocessing metadata:

```text
models/lstm_hurdle/
  classifier.pt
  regressor.pt
  static_stats.pt
  config.json
  data_meta.json
  all_user_ids.npy
```

Если `FINAL_SEEDS` содержит больше одного seed, дополнительные веса сохраняются в `models/lstm_hurdle/seeds/<seed>/`.

In [9]:
def save_checkpoint(model, path, task, seed):
    payload = {
        "architecture": "HybridLSTM-v2",
        "task": task,
        "seed": int(seed),
        "model_kwargs": MODEL_KWARGS,
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    }
    torch.save(payload, path)


def save_artifacts():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    primary_seed, primary_classifier, primary_regressor = final_models[0]

    save_checkpoint(primary_classifier, MODEL_DIR / "classifier.pt", "classifier", primary_seed)
    save_checkpoint(primary_regressor, MODEL_DIR / "regressor.pt", "regressor", primary_seed)

    torch.save(
        {
            "mean": final_static_stats[0].cpu(),
            "std": final_static_stats[1].cpu(),
        },
        MODEL_DIR / "static_stats.pt",
    )

    shutil.copy2(DATA_DIR / "meta.json", MODEL_DIR / "data_meta.json")
    shutil.copy2(DATA_DIR / "all_user_ids.npy", MODEL_DIR / "all_user_ids.npy")

    if len(final_models) > 1:
        for seed, classifier, regressor in final_models[1:]:
            seed_dir = MODEL_DIR / "seeds" / str(seed)
            seed_dir.mkdir(parents=True, exist_ok=True)
            save_checkpoint(classifier, seed_dir / "classifier.pt", "classifier", seed)
            save_checkpoint(regressor, seed_dir / "regressor.pt", "regressor", seed)

    meta_bytes = (DATA_DIR / "meta.json").read_bytes()
    config = {
        "architecture": "HybridLSTM-v2",
        "data_format_version": META["format_version"],
        "data_meta_sha256": hashlib.sha256(meta_bytes).hexdigest(),
        "model_kwargs": MODEL_KWARGS,
        "optimizer": "AdamW",
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "best_classifier_epoch": BEST_CLASSIFIER_EPOCH,
        "best_regressor_epoch": BEST_REGRESSOR_EPOCH,
        "holdout_cutoff": HOLDOUT_CUTOFF,
        "holdout_rmsle": HOLDOUT_RMSLE,
        "final_seeds": FINAL_SEEDS,
        "prediction_formula": "sigmoid(classifier_logit) * clamp(regressor_log, min=0) in log1p-space",
        "sequence_input_size": SEQ_INPUT_SIZE,
        "short_summary_size": SHORT_SUMMARY_SIZE,
        "static_input_size": STATIC_INPUT_SIZE,
        "static_feature_count_on_disk": len(STATIC_FEATURES),
    }
    with open(MODEL_DIR / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("saved model artifacts to:", MODEL_DIR)


save_artifacts()

saved model artifacts to: /Users/pinta/Dev/E-CUP-2026/models/lstm_hurdle


## Inference + submission

Если несколько seed, усредняем **`pred_log`**, а не raw GMV.

In [10]:
all_pred_logs = []
submission_user_ids = None

for seed, classifier, regressor in final_models:
    user_ids, logits, _ = predict_raw(
        classifier,
        INFERENCE_CUTOFF,
        final_static_stats,
        with_target=False,
    )
    user_ids_reg, reg_log, _ = predict_raw(
        regressor,
        INFERENCE_CUTOFF,
        final_static_stats,
        with_target=False,
    )
    np.testing.assert_array_equal(user_ids, user_ids_reg)

    pred_log, _ = make_prediction(logits, reg_log)
    all_pred_logs.append(pred_log)
    submission_user_ids = user_ids

ensemble_pred_log = np.mean(np.stack(all_pred_logs, axis=0), axis=0)
pred = np.expm1(ensemble_pred_log)

submission = pd.read_csv(PROJECT_ROOT / "data" / "sample_submit.csv")
pred_map = pd.Series(pred, index=submission_user_ids)
submission["predict"] = submission["user_id"].map(pred_map)

if submission["predict"].isna().any():
    missing = int(submission["predict"].isna().sum())
    raise AssertionError(f"Не найден prediction для {missing} users из sample_submit")
if (submission["predict"] < 0).any():
    raise AssertionError("Negative predictions are forbidden")

out_path = SUBMISSION_DIR / "lstm_hurdle_v2.csv"
submission.to_csv(out_path, index=False)

print("saved:", out_path)
print("rows:", len(submission))
print("exact zeros:", f"{(submission['predict'] == 0).mean():.2%}")
display(submission.head())

saved: /Users/pinta/Dev/E-CUP-2026/submissions/lstm_hurdle_v2.csv
rows: 250000
exact zeros: 0.00%


,user_id,predict
0,2,2.190423
1,7,56.363895
2,15,4.819376
3,18,124.306396
4,23,0.338693


## Reload smoke-test

Проверяем, что сохранённые веса действительно можно загрузить в новый объект модели. Это ловит ситуацию «submission получили одной моделью, а в `models/` лежит другая».

In [11]:
def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:  # старый PyTorch
        return torch.load(path, map_location=map_location)


def load_saved_model(path):
    checkpoint = safe_torch_load(path, map_location=DEVICE)
    model = HybridLSTM(**checkpoint["model_kwargs"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    return model


saved_stats = safe_torch_load(MODEL_DIR / "static_stats.pt", map_location="cpu")
loaded_stats = (saved_stats["mean"], saved_stats["std"])
loaded_classifier = load_saved_model(MODEL_DIR / "classifier.pt")
loaded_regressor = load_saved_model(MODEL_DIR / "regressor.pt")

# Сравниваем небольшой prefix inference, чтобы не прогонять весь test второй раз.
smoke_dataset = HybridDataset(INFERENCE_CUTOFF, with_target=False)
smoke_loader = DataLoader(smoke_dataset, batch_size=min(256, BATCH_SIZE), shuffle=False)
smoke_batch = next(iter(smoke_loader))
sequence, static, user_index, history_length, _ = unpack_batch(smoke_batch, with_target=False)
static = normalize_static(static, loaded_stats)

with torch.no_grad():
    cls_out = loaded_classifier(sequence, static, user_index, history_length)
    reg_out = loaded_regressor(sequence, static, user_index, history_length)

assert torch.isfinite(cls_out).all()
assert torch.isfinite(reg_out).all()
print("reload smoke-test passed:", len(cls_out), "rows")

reload smoke-test passed: 256 rows


## Что смотреть после запуска

1. Сравнить новый `holdout RMSLE` с прежними `1.679... / 1.697...` — главное, чтобы изменения не ухудшили temporal holdout.
2. Посмотреть выбранные epoch counts; если optimum упёрся в 6 или 16 — расширить только соответствующую границу.
3. Если user embedding ухудшает holdout, первый ablation — `USER_EMBED_DIM=0`/убрать user branch. Не надо сразу выбрасывать mask/packing и short-history features вместе с ним.
4. Если хочется ещё один LSTM-only эксперимент после этого — следующий осмысленный шаг: **joint shared encoder + final log-space loss**, а не ещё один threshold.